# Representational Geometry as a Fidelity Metric for Connectome-Constrained Neural Emulations
**Experiment 4b: Perturbation-Sensitivity Sweep | Michael Zhou | Georgia Tech ECE**

## Why this experiment exists

Experiment 4 aborted at its precision guard. At the Flyvis prior perturbation
(`BIAS_NOISE = 0.05`), the untrained connectome-constrained mean RDM has an off-diagonal
span of 1.66×10⁻⁸ against a float32 round-off floor of 1.93×10⁻⁷ — **the floor is eleven
times the signal** (span/floor = 0.09×). A cancellation-free cross-check (Euclidean on
L2-normalized vectors) gives per-model Kendall τ = 1.0000 against cosine for all 150
models: both metrics agree exactly, and what they agree on is rounding.

Untrained networks at the prior produce population responses so nearly identical across
the twelve directions that the distances between them are unresolvable. No statistic
computed from those RDMs — Spearman, Kendall, permutation — carries information.

That is a finding, not a dead end. It raises a question Experiment 4 could not ask:
**does directional geometry appear at *any* perturbation scale, and if so, is it a property
of the connectome or merely of large perturbations?**

## What this notebook measures

For each `BIAS_NOISE` value in a sweep, and for all three conditions (untrained CC,
syn-shuffled, sign-shuffled):

1. **Resolvability.** The RDM's off-diagonal span against the float32 round-off floor
   implied by the response magnitude. This is a **gate**, not a metric: nothing else is
   reported until it clears.

2. **Circularity.** Spearman r between the mean RDM and the circular-distance reference
   min(|i−j|, 12−|i−j|). This asks the question directly — *does the network order
   directions by angle?* — against an explicit reference rather than through the
   degenerate von Mises proxy (see Experiment 3, where that proxy proved 97.8% circular).

3. **The control.** Whether CC gains circular structure that the shuffled conditions do
   **not**.

4. **Regime diagnostics.** Response magnitude, stability rejections, and metric agreement
   (cosine vs Euclidean-normalized) at every level.

## The control is the point

Raising `BIAS_NOISE` makes responses larger. Larger responses produce larger RDM spans.
Larger spans clear the guard. **This is trivially true and says nothing about wiring.**

If CC, syn-shuffle, and sign-shuffle all become resolvable together and all become
circular together, the sweep has demonstrated that big perturbations produce big responses.
It has not demonstrated a connectome prior. The quantity of interest is therefore not
CC's circularity but the **CC-minus-shuffle difference** in circularity, evaluated only at
scales where all three are resolvable.

Without this control, a perturbation sweep is a machine for manufacturing whatever answer
one is looking for. The notebook reports the difference explicitly at every resolvable
level and states when it is absent.

## What raising the noise costs

`BIAS_NOISE = 0.05` perturbs around the Flyvis prior, which initializes
`nodes_bias ~ N(0.5, 0.05)`. At σ = 0.05 the perturbation is the same scale as the prior's
own spread. At σ = 0.8 it is sixteen times that. At σ = 3.2, "untrained connectome-
constrained network" no longer means *the connectome at initialization* — it means *the
connectome with large random biases*, which is a different object and may not license any
claim about wiring at all.

**There is no principled cutoff, and this notebook does not invent one.** It reports
response magnitudes and stability-rejection counts at every level so that the line can be
drawn deliberately rather than discovered in review. Where that line belongs is a modeling
judgment, and it should be argued for in the text rather than buried in a threshold.

Only `BIAS_NOISE` varies across the sweep. `TIME_CONST_NOISE` and `SYN_STRENGTH_NOISE` are
held fixed, so any change is attributable to one axis.

## The three outcomes, and what each licenses

**The guard never clears within a defensible range.**
→ Untrained networks have no measurable representational geometry. Training *creates* the
directional structure rather than amplifying a wiring prior. This inverts Experiment 4's
original conclusion and is the strongest of the three results, because it is a claim about
the connectome that the original experiment could not have tested — it never checked
whether its RDMs were resolvable.

**The guard clears, and CC circularity exceeds both shuffles.**
→ The connectome prior is real but requires perturbation to become visible. Report the
threshold scale, the CC-vs-shuffle gap at that scale, and the response magnitudes, so a
reader can judge whether the network is still in a defensible regime.

**The guard clears, and CC circularity matches the shuffles.**
→ Perturbation magnitude, not wiring, produced the geometry. There is no prior. This is
the null hypothesis Experiment 4 could never have rejected or accepted, because it
reported statistics on matrices that carried no rank order.

Only the first two support any claim about the connectome. All three are publishable.

## A note on the guard's threshold

The resolvability criterion (span ≥ 10× the round-off floor) is a **heuristic, not a
derived threshold.** The floor uses `max|response|` across all models and stimuli, so a
single outlier inflates it for the whole ensemble; conversely, averaging N RDMs suppresses
noise by ~√N, which the floor does not credit. The verdict is informative when lopsided
(well under 1×, or well over 100×) and uninformative when marginal.

The **cosine-vs-Euclidean cross-check does not depend on this constant.** Per-model Kendall
τ between the two metrics' rank orders is the direct test: τ ≈ 1 means the metric is
irrelevant and the ordering is real; τ ≈ 0 means the ordering is arbitrary. Where the two
disagree, no threshold is needed to know the RDM is noise.

## Configuration

**Stimuli:** 12 ON moving edge directions (0°–330°, 30° increments), speed=19, matching
Experiments 1 and 4 exactly.

**Metrics:** float64 cosine RDM (chord form, `1 − cos θ = ‖â − b̂‖²/2`);
Euclidean-normalized RDM (cancellation-free cross-check); resolvability ratio;
Spearman r against the circular-distance reference.

**Reproducibility:** seed=42 (`numpy`, `torch`, `torch.cuda`);
`torch.use_deterministic_algorithms(True)`.

**Runtime:** ~2.6 s per forward pass × 13 passes per model × N models × 3 conditions × the
number of noise levels. A pilot (`N_MODELS = 5`, four levels) is ≈ 35 minutes on a T4; the
version of record (`N_MODELS = 20`, six levels) is ≈ 3.5 hours.

**Note on scale:** the guard arithmetic implies the RDM span must grow by **more than two
orders of magnitude** to clear the threshold, because raising `BIAS_NOISE` also raises the
response magnitude and therefore the floor. A 100× larger span reaches only ratio ≈ 8.6×.
It is entirely possible the guard never clears within any defensible perturbation range.
That would be the result.

**References:**
- Lappalainen et al. 2024. Connectome-constrained networks predict neural activity across the fly visual system. *Nature* 634, 1132–1140. https://www.nature.com/articles/s41586-024-07939-3
- Maisak et al. 2013. A directional tuning map of Drosophila elementary motion detectors. *Nature* 500, 212–216. https://doi.org/10.1038/nature12320
- Kriegeskorte et al. 2008. Representational similarity analysis. *Frontiers in Systems Neuroscience* 2:4. https://www.frontiersin.org/journals/systems-neuroscience/articles/10.3389/neuro.06.004.2008/full
- Nili et al. 2014. A toolbox for representational similarity analysis. *PLOS Computational Biology* 10(4): e1003553. https://doi.org/10.1371/journal.pcbi.1003553
- Brunton et al. 2026. The digital sphinx: Can a worm brain control a fly body? *bioRxiv*. https://www.biorxiv.org/content/10.64898/2026.03.20.713233v1

## Setup

Install Flyvis and download pretrained models. Select **Runtime → Change runtime type → GPU**
before running. Tested on T4 GPU (14.56 GiB).

In [ ]:
# Install packages
!git clone https://github.com/TuragaLab/flyvis.git
%cd /content/flyvis
!pip install -e .[examples]
!flyvis download-pretrained

Cloning into 'flyvis'...
remote: Enumerating objects: 5230, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 5230 (delta 170), reused 148 (delta 148), pack-reused 5028 (from 3)
Receiving objects: 100% (5230/5230), 188.38 MiB | 15.28 MiB/s, done.
Resolving deltas: 100% (2679/2679), done.
/content/flyvis
Obtaining file:///content/flyvis
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 63.8 MB/s

## Experiment

**Why this exists.** Experiment 4 aborted at its precision guard: the untrained CC mean
RDM spans 1.66×10⁻⁸ against a float32 round-off floor of 1.93×10⁻⁷ (ratio 0.09×). The
original's own limitations section proposed larger perturbations as the remedy. This
notebook tests that proposal, one parameter group at a time.

**Design.** Two sweeps, N=5 models per condition, three conditions each (CC, syn-shuffle,
sign-shuffle). Only one parameter varies per sweep; the others are held at their prior
values, so any change is attributable to one axis.

- **Bias-noise sweep:** `BIAS_NOISE` ∈ {0.05, 0.2, 0.8, 3.2}, with
  `SYN_STRENGTH_NOISE = 0.002` and `TIME_CONST_NOISE = 0.005` fixed.
- **Synapse-strength sweep:** `SYN_STRENGTH_NOISE` ∈ {0.002, 0.008, 0.032, 0.128}, with
  `BIAS_NOISE = 0.05` and `TIME_CONST_NOISE = 0.005` fixed.

**Note on ensemble size.** N=5 here versus N=50 in Experiment 4. At the shared
configuration the sweep reports span/floor = 0.05× against Experiment 4's 0.09×;
averaging fewer RDMs suppresses less noise, and the guard's floor does not credit
ensemble size. This is one reason the 10× margin is a heuristic. The per-model
cross-check is unaffected: **Kendall τ = 1.0000 at every level in every condition.**

**Stimuli:** 12 ON moving edge conditions, matching Experiments 1 and 4 exactly.

**What is recorded at every level:**
- **Resolvability ratio** — RDM off-diagonal span ÷ float32 round-off floor
  (`eps_f32 × max|response|`). This is a **gate**: circularity is reported only where it
  clears. Nothing is computed on an unresolvable matrix.
- **Circularity** — Spearman r between the mean RDM and min(|i−j|, 12−|i−j|). Asks
  directly whether the network orders directions by angle, against an *explicit*
  reference rather than the degenerate von Mises proxy (see Experiment 3).
- **The control** — whether CC gains circular structure the shuffles do **not**. If all
  three become resolvable and circular together, large perturbations produced large
  responses, not a wiring prior. Without this, a sweep is a machine for manufacturing
  whichever answer is sought.
- **Regime diagnostics** — response magnitude, stability rejections, metric agreement,
  and the **fraction of cell-type pairs whose `edges_syn_strength` is clamped to zero**.

**The pruning diagnostic is load-bearing, and it points the opposite way from intuition.**
`edges_syn_strength` (class `SynapseCountScaling`) is one scaling factor per cell-type pair
— 604 in total — initialized as **`0.01 / mean_syn_count`**, i.e. **inverse** to synapse
count (verified: Pearson r = 1.0000, max residual 2.4×10⁻⁷). The *densest* connections
therefore carry the *smallest* factors, and `clamp = non_negative` silences them first.

At the prior (σ = 0.002), **8.1% of the 604 cell-type pairs are already zeroed (range
5.6–10.3%, n = 50 seeds)** — every untrained CC model in this work is sparsified as well as
perturbed. The bias is severe: 40.7% of the lowest-factor decile (11–144 synapses per
instance) is silenced, against 0% of the top four deciles (0.27–1.5); Mann–Whitney
p = 3×10⁻²². Median `mean_syn_count` of silenced pairs is 11.4, against a population median
of 1.99.

**But the bias is a small-σ phenomenon, and this is where the regime ends.** As σ grows the
noise exceeds even the largest scaling factor (3.7×10⁻²), and `clamp = non_negative`
becomes a coin flip on the sign of a mean-zero Gaussian. At σ = 0.128 **every decile is
silenced at ~48%** (decile 1: 40.7→48.9%; decile 10: 0.0→47.0%), the median synapse count
of silenced pairs falls from 11.4 to 2.0, and 48.2% of pairs are gone. The manipulation
changes character along the sweep: at the prior it removes the connectome's densest
projections; at the top it deletes half of them **at random**.

Without this column, a resolvable RDM at the top of the sweep would read as evidence of a
wiring prior when it is evidence of a network with half its cell-type projections deleted.

*(For reference: task training also zeroes 31/604 factors, so the clamp is an active
mechanism in Flyvis, not a dormant rail. But training selects a nearly disjoint set —
overlap 3, expected 2.5 — with no detectable density bias, p = 0.080. The bias is specific
to this perturbation protocol.)*

**What raising the noise costs.** `BIAS_NOISE = 0.05` perturbs around a prior that
initializes `nodes_bias ~ N(0.5, 0.05)` — the perturbation is the same scale as the prior's
own spread. At σ = 3.2 it is sixty times that, and "untrained connectome-constrained
network" no longer means *the connectome at initialization*. There is no principled cutoff
and this notebook does not invent one; it reports response magnitudes, rejection counts,
and silencing fractions so the line can be drawn deliberately rather than discovered in
review.

**The three outcomes, stated in advance:**

| outcome | what it licenses |
|---|---|
| Guard never clears within a defensible range | Untrained networks have no measurable representational geometry. Training does not amplify a wiring prior. |
| Guard clears, CC circularity exceeds both shuffles | A wiring prior exists but requires perturbation to become visible. Report threshold, gap, and regime together. |
| Guard clears, CC matches the shuffles | Perturbation magnitude, not wiring, produced the geometry. No prior. |

Only the first two support any claim about the connectome. All three are reportable.

**Metrics:** float64 cosine RDM (chord form, `1 − cos θ = ‖â − b̂‖² / 2`);
Euclidean-normalized RDM as a cancellation-free cross-check; resolvability ratio;
Spearman r against the circular-distance reference.

**Reproducibility:** seed=42 (`numpy`, `torch`, `torch.cuda`);
`torch.use_deterministic_algorithms(True)`.

**Runtime:** ~2.6 s per forward pass × 13 passes × N models × 3

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — exp4_perturbation_sweep: SYNAPSE-STRENGTH AXIS
#
# Does untrained connectome geometry become measurable when the parameter
# the connectome actually constrains is perturbed?
#
# CONTEXT. Experiment 4 aborted at the precision guard: at the Flyvis prior
# the untrained CC mean RDM spans 1.66e-08 against a float32 round-off floor
# of 1.93e-07 (ratio 0.09x). A cancellation-free cross-check (euclidean on
# L2-normalised vectors) gives per-model Kendall tau = 1.0000 against cosine
# for all 150 models: both metrics see the same thing, and it is rounding.
#
# A first sweep varied BIAS_NOISE (0.05 -> 3.2) and found CC's RDM span
# FALLING monotonically -- 1.03e-08, 8.71e-09, 3.23e-09, 2.53e-10 -- while
# response magnitude rose from 1.59 to 11.08. nodes_bias is a per-node
# additive constant; a large common offset drives all twelve population
# vectors toward the same bias-dominated direction. Cosine distance is
# scale-invariant, so it does not see that the vectors grew, only that the
# angle between them shrank. Bias noise does not fail to reveal directional
# geometry -- it destroys it. That axis is foreclosed.
#
# THIS SWEEP varies SYN_STRENGTH_NOISE instead. edges_syn_strength is what
# the connectome constrains. If CC's span rises with it while the shuffles'
# do not, there is a wiring prior. If CC stays flat, the connectome imposes
# no measurable directional geometry under perturbation of its own free
# parameters -- and "training creates the geometry" is earned rather than
# asserted.
#
# WHAT VARIES: only SYN_STRENGTH_NOISE. BIAS_NOISE and TIME_CONST_NOISE are
# held fixed at their prior values.
#
# THE CONTROL: at every resolvable level, does CC gain circular structure
# that the SHUFFLES do not? If all three become circular together, large
# perturbations produced large responses -- not a wiring prior. Without this
# check the sweep is a machine for manufacturing the answer.
#
# WATCH THE PRUNING. edges_syn_strength initialises to 0.01 * syn_count and
# is clamped at min=0.0. At large syn_noise the Gaussian perturbation exceeds
# most weights and clamps them to exactly zero. That is PRUNING, not
# perturbation. The fraction of zeroed edges is printed at every level. Where
# it is large, this is a different experiment and the interpretation stops.
# ═══════════════════════════════════════════════════════════════════

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr, kendalltau

import flyvis
from flyvis.datasets.moving_bar import MovingEdge
from flyvis.network import Network
from flyvis.utils.activity_utils import LayerActivity

# ── CONFIG ──────────────────────────────────────────────────────────
# SYN_STRENGTH_NOISE is the swept axis. Its prior value is 0.002, the first
# level below; each subsequent level is 4x.
NOISE_LEVELS = [0.002, 0.008, 0.032, 0.128]
N_MODELS     = 5                    # pilot; 20 for the version of record

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)

OVERFLOW_LIMIT = 1e6
MAX_ATTEMPTS   = 20
FLOAT32_EPS    = float(np.finfo(np.float32).eps)   # 1.19e-07

# RESOLVE_MARGIN is a heuristic, not a derived threshold. 10x was chosen for
# headroom. The floor uses max|response| over ALL models and stimuli, so one
# outlier inflates it; averaging N RDMs suppresses noise by ~sqrt(N), which the
# floor does not credit. Trust the verdict only when lopsided (<<1x or >>100x).
RESOLVE_MARGIN = 10.0

BIAS_NOISE       = 0.05    # HELD FIXED at the Flyvis prior value
TIME_CONST_NOISE = 0.005   # HELD FIXED
# NOTE: there is deliberately no SYN_STRENGTH_NOISE constant. The swept value
# is passed into make_untrained_cc() as `syn_noise`. A module-level constant
# here would be dead code, and a future "fix" that used it would silently stop
# the sweep from sweeping.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANGLES = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]
N_DIR  = len(ANGLES)

OUT = Path("/content")
(OUT / "results").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}   n_models={N_MODELS}")
print(f"Swept axis: SYN_STRENGTH_NOISE = {NOISE_LEVELS}")
print(f"Held fixed: BIAS_NOISE = {BIAS_NOISE}, TIME_CONST_NOISE = {TIME_CONST_NOISE}\n")


# ── DISTANCES (float64; chord form never cancels) ────────────────────
def cosine_distance_stable(a, b):
    """1 - cos(theta) via ||a_hat - b_hat||^2 / 2, evaluated in float64.

    The float64 cast is the substantive fix: scipy.spatial.distance.cosine
    preserves input dtype, and float32 inputs destroy distances at the 1e-8
    scale these networks produce. The chord form is a sum of squares and
    costs nothing.
    """
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return np.nan
    d = a / na - b / nb
    return float(d @ d) / 2.0


def euclidean_normalized(a, b):
    """Cancellation-free cross-check: monotone in the angle, no subtraction from 1."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return np.nan
    return float(np.linalg.norm(a / na - b / nb))


def build_rdm(P, metric="cosine"):
    P = np.asarray(P, dtype=np.float64)
    if not np.isfinite(P).all():
        raise ValueError("non-finite population vector")
    fn = cosine_distance_stable if metric == "cosine" else euclidean_normalized
    n = P.shape[0]
    R = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = fn(P[i], P[j])
            if np.isnan(d):
                raise ValueError(f"zero-norm population vector at stimulus {i} or {j}")
            R[i, j] = R[j, i] = d
    return R


def upper(M):
    return M[np.triu_indices_from(M, k=1)]


def circular_reference(n=N_DIR):
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    d = np.abs(i - j)
    return np.minimum(d, n - d).astype(float)


def resolvability(rdm, pop_matrix):
    """The gate. Nothing is reported until this clears."""
    off = upper(rdm)
    span = float(off.max() - off.min())
    floor = FLOAT32_EPS * float(np.abs(pop_matrix).max())
    ratio = span / floor if floor > 0 else np.inf
    return span, floor, ratio, ratio >= RESOLVE_MARGIN


# ── NETWORKS (syn_noise is the swept parameter) ──────────────────────
def make_untrained_cc(seed, syn_noise):
    """Untrained CC network. Returns (net, frac_zeroed).

    frac_zeroed is the fraction of edges_syn_strength clamped to exactly 0.0
    by the min=0.0 clamp. At large syn_noise this is PRUNING, not perturbation,
    and the sweep's interpretation must stop where it becomes large.
    """
    rng = np.random.default_rng(seed)
    net = Network()
    with torch.no_grad():
        net.nodes_bias.data += torch.tensor(
            rng.normal(0, BIAS_NOISE, size=net.nodes_bias.shape),
            dtype=torch.float32)                                    # HELD FIXED

        net.nodes_time_const.data = torch.clamp(
            net.nodes_time_const.data + torch.tensor(
                rng.normal(0, TIME_CONST_NOISE, size=net.nodes_time_const.shape),
                dtype=torch.float32), min=0.001)                    # HELD FIXED

        raw = net.edge_params.syn_strength.raw_values
        raw.data = torch.clamp(raw.data + torch.tensor(
            rng.normal(0, syn_noise, size=raw.shape),               # <- SWEPT
            dtype=torch.float32), min=0.0)

        frac_zeroed = float((raw.data == 0).float().mean().item())

    return net.to(DEVICE), frac_zeroed


def apply_syn_shuffle(net, rng):
    with torch.no_grad():
        raw = net.edge_params.syn_strength.raw_values
        v = raw.data.cpu().numpy().copy()
        pos, neg = np.where(v > 0)[0], np.where(v <= 0)[0]
        if len(pos) > 1: v[pos] = rng.permutation(v[pos])
        if len(neg) > 1: v[neg] = rng.permutation(v[neg])
        raw.data = torch.tensor(v, dtype=torch.float32).to(DEVICE)
    return net


def apply_sign_shuffle(net, rng):
    with torch.no_grad():
        sp = net.edge_params.sign.raw_values
        v = sp.data.cpu().numpy().copy()
        n_exc, n_inh = int((v > 0).sum()), int((v < 0).sum())
        sp.data = torch.tensor(
            rng.permutation(np.array([1.0]*n_exc + [-1.0]*n_inh, dtype=np.float32)),
            dtype=torch.float32).to(DEVICE)
        raw = net.edge_params.syn_strength.raw_values
        s = raw.data.cpu().numpy().copy()
        pos, neg = np.where(s > 0)[0], np.where(s <= 0)[0]
        if len(pos) > 1: s[pos] = rng.permutation(s[pos])
        if len(neg) > 1: s[neg] = rng.permutation(s[neg])
        raw.data = torch.tensor(s, dtype=torch.float32).to(DEVICE)
    return net


# ── SIMULATION ───────────────────────────────────────────────────────
def get_cell_types(net):
    return [ct.decode() if isinstance(ct, bytes) else ct
            for ct in net.connectome.unique_cell_types[:]]


def get_population_vector(net, stim, dt, cell_types):
    if stim.dim() == 2:
        stim = stim.unsqueeze(1)
    s0 = net.fade_in_state(1.0, dt, stim[[0]])
    with torch.no_grad():
        resp = net.simulate(stim[None], dt, initial_state=s0).cpu()
    la = LayerActivity(resp, net.connectome, keepref=True)
    pv = np.array([la.central[ct].squeeze().numpy().max() for ct in cell_types],
                  dtype=np.float64)
    pv = np.clip(pv, -OVERFLOW_LIMIT, OVERFLOW_LIMIT)
    del resp, la
    torch.cuda.empty_cache()
    return pv


def is_stable(net, stim, dt):
    """Deliberately does NOT swallow exceptions. A CUDA OOM or shape error is
    not 'instability', and the original's bare try/except would have counted it
    as one, silently."""
    st = stim.unsqueeze(1) if stim.dim() == 2 else stim
    s0 = net.fade_in_state(1.0, dt, st[[0]])
    out = net.simulate(st[None], dt, initial_state=s0)
    o = out.cpu().numpy()
    return bool(torch.all(torch.isfinite(out)) and np.all(np.abs(o) < OVERFLOW_LIMIT))


def run_condition(make_fn, n_models, dataset, stim_idx, dt, label):
    """make_fn(seed) -> (net, frac_zeroed)."""
    check = dataset[stim_idx[0]]
    if not isinstance(check, torch.Tensor):
        check = torch.tensor(check, dtype=torch.float32)
    check = check.to(DEVICE)

    pops, rc, re_, zeroed = [], [], [], []
    accepted = seed = rejected = 0
    cts = None

    while accepted < n_models and seed < n_models * MAX_ATTEMPTS:
        net, fz = make_fn(seed); seed += 1
        if not is_stable(net, check, dt):
            rejected += 1; del net; torch.cuda.empty_cache(); continue
        if cts is None:
            cts = get_cell_types(net)

        pv_list, ok = [], True
        for si in stim_idx:
            s = dataset[si]
            if not isinstance(s, torch.Tensor):
                s = torch.tensor(s, dtype=torch.float32)
            pv = get_population_vector(net, s.to(DEVICE), dt, cts)
            if not np.isfinite(pv).all():
                ok = False; break
            pv_list.append(pv)
        if not ok:
            rejected += 1; del net; torch.cuda.empty_cache(); continue

        pm = np.stack(pv_list, 0)
        pops.append(pm)
        rc.append(build_rdm(pm, "cosine"))
        re_.append(build_rdm(pm, "euclidean_normalized"))
        zeroed.append(fz)
        accepted += 1
        print(f"      {label}: {accepted}/{n_models}", flush=True)
        del net; torch.cuda.empty_cache()

    return (np.stack(pops, 0), np.stack(rc, 0), np.stack(re_, 0),
            rejected, float(np.mean(zeroed)))


# ── SWEEP ────────────────────────────────────────────────────────────
dataset = MovingEdge(offsets=[-10, 11], intensities=[0, 1], speeds=[19],
                     height=80, post_pad_mode="continue",
                     t_pre=1.0, t_post=1.0, dt=1/200, angles=ANGLES)
on_idx = [i for i, r in dataset.arg_df.iterrows() if r["intensity"] == 1]
assert len(on_idx) == N_DIR

circ = circular_reference()
rows = []

for sn in NOISE_LEVELS:
    print("=" * 78)
    print(f"SYN_STRENGTH_NOISE = {sn}   (BIAS_NOISE fixed at {BIAS_NOISE})")
    print("=" * 78)

    conds = {
        "CC":        lambda s, sn=sn: make_untrained_cc(s, sn),
        "Rand-syn":  lambda s, sn=sn: (
            apply_syn_shuffle(make_untrained_cc(s, sn)[0],
                              np.random.default_rng(s + 10_000)),
            make_untrained_cc(s, sn)[1]),
        "Rand-sign": lambda s, sn=sn: (
            apply_sign_shuffle(make_untrained_cc(s, sn)[0],
                               np.random.default_rng(s + 20_000)),
            make_untrained_cc(s, sn)[1]),
    }

    level = {}
    for name, fn in conds.items():
        pops, rc, re_, rej, fz = run_condition(fn, N_MODELS, dataset, on_idx,
                                               dataset.dt, name)
        mean_cos, mean_euc = rc.mean(0), re_.mean(0)
        span, floor, ratio, ok = resolvability(
            mean_cos, pops.reshape(-1, pops.shape[-1]))

        taus = np.array([kendalltau(upper(rc[i]), upper(re_[i])).statistic
                         for i in range(rc.shape[0])])
        tau_mean = kendalltau(upper(mean_cos), upper(mean_euc)).statistic

        # circularity reported ONLY where resolvable
        r_circ = spearmanr(upper(mean_cos), upper(circ)).statistic if ok else np.nan

        level[name] = dict(span=span, floor=floor, ratio=ratio, ok=ok,
                           r_circ=r_circ, rej=rej,
                           resp=float(np.abs(pops).max()),
                           tau_med=float(np.median(taus)),
                           tau_mean=float(tau_mean),
                           frac_zeroed=fz)

        flag   = "RESOLVABLE " if ok else "below floor"
        circ_s = f"{r_circ:+.3f}" if ok else "  n/a "
        print(f"  {name:10s} span {span:.2e}  floor {floor:.2e}  "
              f"ratio {ratio:6.2f}x  {flag}  r(circ) {circ_s}  "
              f"|resp| {level[name]['resp']:.2f}  rej {rej}  "
              f"tau_pm {level[name]['tau_med']:.3f}  zeroed {fz:.1%}")

    # PRUNING WARNING -- the sweep leaves the perturbation regime here.
    fz_cc = level["CC"]["frac_zeroed"]
    if fz_cc > 0.10:
        print(f"\n  ** {fz_cc:.1%} of edges_syn_strength clamped to exactly 0.0.")
        print("     This is PRUNING, not perturbation. 'Untrained CC' no longer")
        print("     means the connectome with perturbed weights. Interpret with care.")

    # THE CONTROL
    if all(level[c]["ok"] for c in level):
        d_syn  = level["CC"]["r_circ"] - level["Rand-syn"]["r_circ"]
        d_sign = level["CC"]["r_circ"] - level["Rand-sign"]["r_circ"]
        print(f"\n  All resolvable. CC circularity advantage: "
              f"vs syn {d_syn:+.3f}, vs sign {d_sign:+.3f}")
        if max(d_syn, d_sign) < 0.05:
            print("  -> CC does NOT exceed the shuffles. Perturbation magnitude,")
            print("     not wiring, produced any geometry here.")
        else:
            print("  -> CC exceeds the shuffles. Consistent with a wiring prior")
            print("     that requires perturbation to become visible.")
    elif not level["CC"]["ok"]:
        print("\n  CC not resolvable. No statistic reported at this scale.")

    rows.append((sn, level))
    print()


# ── SUMMARY ──────────────────────────────────────────────────────────
print("=" * 78); print("SUMMARY  (swept axis: SYN_STRENGTH_NOISE)"); print("=" * 78)
print(f"  {'syn_noise':>10} {'CC ratio':>9} {'CC span':>11} {'CC |resp|':>10} "
      f"{'CC zeroed':>10} {'CC r(circ)':>11} {'syn r(circ)':>12} {'sign r(circ)':>13}")
for sn, lv in rows:
    f = lambda k: (f"{lv[k]['r_circ']:+.3f}" if lv[k]["ok"] else "   n/a")
    print(f"  {sn:>10.3f} {lv['CC']['ratio']:>8.2f}x {lv['CC']['span']:>11.2e} "
          f"{lv['CC']['resp']:>10.2f} {lv['CC']['frac_zeroed']:>9.1%} "
          f"{f('CC'):>11} {f('Rand-syn'):>12} {f('Rand-sign'):>13}")

first_ok = next((sn for sn, lv in rows if lv["CC"]["ok"]), None)
print()
if first_ok is None:
    print("  The guard never cleared across the swept range.")
    print("  Report the swept axis, the span trajectory, the response magnitudes,")
    print("  and the pruning fraction. Do NOT conclude beyond the axis swept: this")
    print("  sweep constrains SYN_STRENGTH_NOISE only. A separate sweep already")
    print("  foreclosed BIAS_NOISE (CC span fell monotonically as noise rose).")
else:
    print(f"  Untrained CC geometry first becomes resolvable at "
          f"SYN_STRENGTH_NOISE = {first_ok}.")
    print("  Whether that constitutes a connectome prior depends on the CC-vs-shuffle")
    print("  comparison at and above that scale, on whether the network remains in a")
    print("  defensible regime, and on how many edges were pruned to zero. Report")
    print("  threshold, gap, response magnitudes, and pruning fraction together.")

print("\n  Caveat, unavoidable: raising SYN_STRENGTH_NOISE moves the network away")
print("  from the Flyvis prior, and the min=0.0 clamp prunes edges rather than")
print("  perturbing them once the noise exceeds typical weights. Response")
print("  magnitudes, rejection counts, and the zeroed fraction are reported so")
print("  that line can be drawn deliberately, not by accident.")

np.savez(OUT / "results" / "exp4_sweep_syn.npz",
         swept_axis=np.array(["SYN_STRENGTH_NOISE"]),
         bias_noise_fixed=np.array([BIAS_NOISE]),
         noise=np.array([r[0] for r in rows]),
         **{f"{c}_{k}": np.array([r[1][c][k] for r in rows])
            for c in ("CC", "Rand-syn", "Rand-sign")
            for k in ("span", "floor", "ratio", "r_circ", "resp",
                      "tau_med", "tau_mean", "rej", "frac_zeroed")})

# ── FIGURE ───────────────────────────────────────────────────────────
noise = np.array([r[0] for r in rows])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

for c, col in [("CC", "k"), ("Rand-syn", "tab:blue"), ("Rand-sign", "tab:red")]:
    ax[0].plot(noise, [r[1][c]["ratio"] for r in rows], "o-", color=col, label=c)
ax[0].axhline(RESOLVE_MARGIN, ls="--", c="gray",
              label=f"resolvability threshold ({RESOLVE_MARGIN:g}x)")
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("SYN_STRENGTH_NOISE"); ax[0].set_ylabel("RDM span / float32 floor")
ax[0].set_title("Is the RDM resolvable?"); ax[0].legend(fontsize=8)

for c, col in [("CC", "k"), ("Rand-syn", "tab:blue"), ("Rand-sign", "tab:red")]:
    ax[1].plot(noise, [r[1][c]["r_circ"] for r in rows], "o-", color=col, label=c)
ax[1].axhline(0, ls=":", c="gray")
ax[1].set_xscale("log")
ax[1].set_xlabel("SYN_STRENGTH_NOISE")
ax[1].set_ylabel("r(mean RDM, circular reference)")
ax[1].set_title("Circular direction structure\n(only where resolvable)")
ax[1].legend(fontsize=8)

# The regime plot: where does perturbation become pruning?
ax[2].plot(noise, [r[1]["CC"]["frac_zeroed"] for r in rows], "o-", color="k")
ax[2].axhline(0.10, ls="--", c="gray", label="10% pruned")
ax[2].set_xscale("log")
ax[2].set_xlabel("SYN_STRENGTH_NOISE")
ax[2].set_ylabel("fraction of edges clamped to 0")
ax[2].set_title("Perturbation or pruning?")
ax[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "figures" / "exp4_sweep_syn.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved /content/results/exp4_sweep_syn.npz and "
      "/content/figures/exp4_sweep_syn.png")

Device: cuda   n_models=5
Swept axis: SYN_STRENGTH_NOISE = [0.002, 0.008, 0.032, 0.128]
Held fixed: BIAS_NOISE = 0.05, TIME_CONST_NOISE = 0.005

SYN_STRENGTH_NOISE = 0.002   (BIAS_NOISE fixed at 0.05)
      CC: 1/5
      CC: 2/5
      CC: 3/5
      CC: 4/5
      CC: 5/5
  CC         span 1.03e-08  floor 1.90e-07  ratio   0.05x  below floor  r(circ)   n/a   |resp| 1.59  rej 0  tau_pm 1.000  zeroed 8.8%
      Rand-syn: 1/5
      Rand-syn: 2/5
      Rand-syn: 3/5
      Rand-syn: 4/5
      Rand-syn: 5/5
  Rand-syn   span 2.47e-07  floor 2.45e-07  ratio   1.01x  below floor  r(circ)   n/a   |resp| 2.06  rej 0  tau_pm 1.000  zeroed 8.8%
      Rand-sign: 1/5
      Rand-sign: 2/5
      Rand-sign: 3/5
      Rand-sign: 4/5
      Rand-sign: 5/5
  Rand-sign  span 3.50e-06  floor 1.95e-06  ratio   1.80x  below floor  r(circ)   n/a   |resp| 16.32  rej 0  tau_pm 1.000  zeroed 8.8%

  CC not resolvable. No statistic reported at this scale.

SYN_STRENGTH_NOISE = 0.008   (BIAS_NOISE fixed at 0.05)
     

In [ ]:
for s in range(5):
    _, fz = make_untrained_cc(s, 0.002)
    print(f"seed {s}: {fz:.1%}")

seed 0: 8.3%
seed 1: 10.3%
seed 2: 8.6%
seed 3: 8.1%
seed 4: 8.9%


In [10]:
# ═══════════════════════════════════════════════════════════════════
# Pruning diagnostic — what does the min=0.0 clamp actually do?
#
# Three questions, in order:
#   (1) How does Flyvis actually initialize edges_syn_strength?
#       (Our Methods says "0.01 × syn_count", but raw.min() = 6.94e-05,
#        which is 144× below 0.01. The formula is wrong somewhere.)
#   (2) What fraction of the 604 cell-type pairs does the clamp silence,
#       across seeds, using the REAL clamp rather than a threshold proxy?
#   (3) Is the deletion biased toward weak connections?
#
# Nothing here trains or simulates. CPU, seconds.
# ═══════════════════════════════════════════════════════════════════

import numpy as np
import torch
import inspect
from flyvis.network import Network

np.set_printoptions(precision=4, suppress=False)


# ── (1) WHAT DOES FLYVIS ACTUALLY DO? ───────────────────────────────
print("=" * 72)
print("(1) INITIALIZATION: is edges_syn_strength really 0.01 × syn_count?")
print("=" * 72)

net = Network()
sp  = net.edge_params.syn_strength
raw = sp.raw_values.data.cpu().numpy()

print(f"  raw_values: shape {raw.shape}, dtype {raw.dtype}")
print(f"    min  {raw.min():.6e}")
print(f"    max  {raw.max():.6e}")
print(f"    mean {raw.mean():.6e}")
print(f"    median {np.median(raw):.6e}")

# Is there a transform between raw_values and the value the network uses?
print(f"\n  type(syn_strength) = {type(sp).__name__}")
print(f"  public attrs: {[a for a in dir(sp) if not a.startswith('_')]}")

for name in ('values', 'value', 'constrained_values', 'transform', 'readout'):
    if hasattr(sp, name):
        v = getattr(sp, name)
        if callable(v):
            print(f"  sp.{name} is callable: {inspect.signature(v) if hasattr(v,'__call__') else ''}")
        else:
            try:
                arr = np.asarray(v.detach().cpu() if torch.is_tensor(v) else v)
                print(f"  sp.{name}: shape {arr.shape}, range {arr.min():.3e} .. {arr.max():.3e}")
                if arr.shape == raw.shape and not np.allclose(arr, raw):
                    print("    ** DIFFERS FROM raw_values — there is a transform. **")
            except Exception as e:
                print(f"  sp.{name}: {e}")

# Find the per-cell-type-pair synapse count (length 604), whatever it's called.
print("\n  searching connectome.edges for a length-604 array:")
e = net.connectome.edges
cand = {}
for a in dir(e):
    if a.startswith('_'):
        continue
    try:
        v = getattr(e, a)[:]
        arr = np.asarray(v)
        if arr.ndim == 1 and arr.shape[0] == raw.shape[0]:
            cand[a] = arr
            print(f"    {a:22s} dtype={arr.dtype}  range {arr.min()} .. {arr.max()}")
    except Exception:
        pass

if not cand:
    print("    (none found at length 604 — syn counts live at synapse resolution)")

# Test the claimed relationship against every candidate.
print("\n  testing  raw ?= 0.01 × <candidate>  :")
for name, arr in cand.items():
    if not np.issubdtype(arr.dtype, np.number):
        continue
    pred = 0.01 * arr.astype(float)
    if pred.max() == 0:
        continue
    ratio = raw / np.where(pred == 0, np.nan, pred)
    ok = np.allclose(raw, pred, rtol=1e-3)
    print(f"    {name:22s} allclose={ok}   raw/pred: median {np.nanmedian(ratio):.4g}, "
          f"std {np.nanstd(ratio):.3g}")
    if ok:
        print(f"      ** raw = 0.01 × {name} CONFIRMED **")

# Is raw proportional to any candidate (allowing a different constant)?
print("\n  testing  raw ?= k × <candidate>  (fitting k):")
for name, arr in cand.items():
    if not np.issubdtype(arr.dtype, np.number) or arr.std() == 0:
        continue
    a = arr.astype(float)
    k = float(np.dot(raw, a) / np.dot(a, a))
    resid = np.abs(raw - k * a).max()
    r = np.corrcoef(raw, a)[0, 1]
    print(f"    {name:22s} k={k:.6e}   max|resid|={resid:.3e}   pearson r={r:.4f}")


# ── (2) THE CLAMP, AS THE CODE ACTUALLY APPLIES IT ──────────────────
print("\n" + "=" * 72)
print("(2) HOW OFTEN DOES min=0.0 SILENCE A CELL-TYPE PAIR?")
print("=" * 72)
print("  Criterion: raw + N(0, sigma) <= 0.0   (what make_untrained_cc does),")
print("  NOT raw < sigma (a threshold proxy, which overestimates).\n")

SIGMA = 0.002
N_SEEDS = 20

fracs, medians_z, medians_k = [], [], []
for s in range(N_SEEDS):
    rng = np.random.default_rng(s)
    noise = rng.normal(0, SIGMA, size=raw.shape)
    zeroed = (raw + noise) <= 0.0
    fracs.append(zeroed.mean())
    if zeroed.any():
        medians_z.append(np.median(raw[zeroed]))
    if (~zeroed).any():
        medians_k.append(np.median(raw[~zeroed]))
    if s < 5:
        print(f"  seed {s}: {zeroed.sum():3d}/604 = {zeroed.mean():.1%}")

fracs = np.array(fracs)
print(f"\n  across {N_SEEDS} seeds:")
print(f"    mean  {fracs.mean():.1%}")
print(f"    range {fracs.min():.1%} .. {fracs.max():.1%}")
print(f"    sd    {fracs.std():.2%}")
print(f"\n  QUOTE THIS: {fracs.mean():.1%} of the 604 cell-type pairs are silenced "
      f"(range {fracs.min():.1%}-{fracs.max():.1%}, n={N_SEEDS} seeds).")


# ── (3) IS THE DELETION BIASED TOWARD WEAK CONNECTIONS? ─────────────
print("\n" + "=" * 72)
print("(3) IS THE PRUNING BIASED TOWARD WEAK CELL-TYPE PAIRS?")
print("=" * 72)

rng = np.random.default_rng(0)
zeroed = (raw + rng.normal(0, SIGMA, size=raw.shape)) <= 0.0

mz, mk = np.median(raw[zeroed]), np.median(raw[~zeroed])
print(f"  seed 0: {zeroed.sum()}/604 silenced")
print(f"    silenced pairs:  median raw {mz:.3e}   max {raw[zeroed].max():.3e}")
print(f"    surviving pairs: median raw {mk:.3e}   min {raw[~zeroed].min():.3e}")
print(f"    ratio of medians: {mk/mz:.1f}x")
print(f"\n  Overlap: silenced reach {raw[zeroed].max():.2e}, survivors go down to "
      f"{raw[~zeroed].min():.2e}.")
print("  So it is a soft filter, not a hard threshold — but the bias is systematic.")

# Averaged across seeds, to make sure seed 0 isn't special.
print(f"\n  across {N_SEEDS} seeds:")
print(f"    median raw of silenced pairs:  {np.mean(medians_z):.3e} "
      f"(sd {np.std(medians_z):.1e})")
print(f"    median raw of surviving pairs: {np.mean(medians_k):.3e} "
      f"(sd {np.std(medians_k):.1e})")
print(f"    ratio: {np.mean(medians_k)/np.mean(medians_z):.1f}x")

# Rank test: is the silenced set drawn from the low tail?
from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(raw[zeroed], raw[~zeroed], alternative='less')
print(f"\n  Mann-Whitney U (silenced < surviving): U={u:.0f}, p={p:.3e}")

# Deciles: what fraction of each strength decile gets silenced?
print("\n  Silencing probability by strength decile (seed-averaged, 20 seeds):")
deciles = np.quantile(raw, np.linspace(0, 1, 11))
counts = np.zeros(10); totals = np.zeros(10)
for s in range(N_SEEDS):
    rng = np.random.default_rng(s)
    z = (raw + rng.normal(0, SIGMA, size=raw.shape)) <= 0.0
    idx = np.clip(np.digitize(raw, deciles[1:-1]), 0, 9)
    for d in range(10):
        m = idx == d
        counts[d] += z[m].sum(); totals[d] += m.sum()

for d in range(10):
    frac = counts[d] / totals[d] if totals[d] else 0.0
    bar = '#' * int(frac * 50)
    print(f"    decile {d+1:2d} "
          f"[{deciles[d]:.2e}, {deciles[d+1]:.2e}]  {frac:5.1%}  {bar}")


# ── SUMMARY ─────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("SENTENCE FOR METHODS (fill from the numbers above)")
print("=" * 72)
print(f"""
  edges_syn_strength is clamped non-negative after Gaussian perturbation
  (sigma = {SIGMA}). The prior's scaling factors span
  {raw.min():.2e} to {raw.max():.2e}, so a sigma = {SIGMA} perturbation pushes a
  substantial fraction negative and the clamp silences them. Across
  {N_SEEDS} seeds, {fracs.mean():.1%} of the 604 cell-type pairs are set to exactly
  zero (range {fracs.min():.1%}-{fracs.max():.1%}). The deletion is not uniform:
  silenced pairs have median scaling factor {np.mean(medians_z):.2e} against
  {np.mean(medians_k):.2e} for surviving pairs ({np.mean(medians_k)/np.mean(medians_z):.0f}x), so the
  perturbation preferentially removes the weakest cell-type connections.
  Untrained connectome-constrained networks are therefore sparsified along
  the weak tail of the connectivity distribution, not merely perturbed.

  ** Verify the initialization claim before writing it: our Methods says
     "edges_syn_strength = 0.01 x syn_count", but raw.min() = {raw.min():.2e},
     which is {0.01/raw.min():.0f}x below 0.01. Section (1) above tests this. **
""")

(1) INITIALIZATION: is edges_syn_strength really 0.01 × syn_count?
  raw_values: shape (604,), dtype float32
    min  6.940063e-05
    max  3.725345e-02
    mean 5.502456e-03
    median 5.021826e-03

  type(syn_strength) = SynapseCountScaling
  public attrs: ['config', 'indices', 'keys', 'parameter', 'raw_values', 'readers', 'semantic_values', 'symmetry_masks']

  searching connectome.edges for a length-604 array:
    (none found at length 604 — syn counts live at synapse resolution)

  testing  raw ?= 0.01 × <candidate>  :

  testing  raw ?= k × <candidate>  (fitting k):

(2) HOW OFTEN DOES min=0.0 SILENCE A CELL-TYPE PAIR?
  Criterion: raw + N(0, sigma) <= 0.0   (what make_untrained_cc does),
  NOT raw < sigma (a threshold proxy, which overestimates).

  seed 0:  48/604 = 7.9%
  seed 1:  45/604 = 7.5%
  seed 2:  52/604 = 8.6%
  seed 3:  44/604 = 7.3%
  seed 4:  48/604 = 7.9%

  across 20 seeds:
    mean  8.1%
    range 5.8% .. 10.3%
    sd    1.01%

  QUOTE THIS: 8.1% of the 604 cell

In [19]:
# ═══════════════════════════════════════════════════════════════════
# Recompute the silencing fractions per-seed, at every sweep level.
#
# WHY. The sweep's frac_zeroed averaged over the 5 ACCEPTED models per
# condition, not over independent seeds. At sigma = 0.002 that gave 8.8%;
# the per-seed loop gives 8.1% (range 5.8-10.3%, n=20). The higher-sigma
# figures (27.2 / 42.6 / 48.2%) came from the same code path and are
# unverified.
#
# Two subtleties the sweep's diagnostic got wrong, and this one does not:
#
#   1. ACCEPTANCE IS NOT RANDOM. At sigma = 0.032 the sweep rejected 2 seeds;
#      at 0.128 it rejected 87 to accept 5. Rejected networks are unstable,
#      and instability correlates with how much was deleted. Averaging over
#      ACCEPTED models therefore conditions on survival and biases the
#      estimate. This script reports BOTH: the unconditional fraction over
#      all seeds, and the fraction restricted to seeds that would pass the
#      stability check. Where they differ, the sweep's number was the
#      conditional one, and the difference is the selection effect.
#
#   2. edges_syn_strength is ONE FACTOR PER CELL-TYPE PAIR (604), not per
#      synapse (1.5M). "Edges pruned" was the wrong unit throughout.
#
# No GPU needed for the unconditional counts (no simulation). The
# stability-conditioned counts DO simulate, so pass --stability to include
# them; expect ~40 s per accepted model.
# ═══════════════════════════════════════════════════════════════════

import numpy as np
import torch
from flyvis.network import Network

SIGMAS  = [0.002, 0.008, 0.032, 0.128]   # the swept SYN_STRENGTH_NOISE levels
N_SEEDS = 50                             # independent draws per level

# The prior's 604 scaling factors. Deterministic — one Network() suffices.
net0 = Network()
raw = net0.edge_params.syn_strength.raw_values.data.cpu().numpy()
assert raw.shape == (604,), raw.shape
del net0

# Per-cell-type-pair mean synapse count, recovered from the verified relation
#   raw = 0.01 / mean_syn_count   (Pearson r = 1.0000, max resid 2.4e-07)
mean_syn = 0.01 / raw

print("=" * 74)
print("SILENCING FRACTION BY SIGMA — unconditional, per-seed")
print("=" * 74)
print(f"  604 cell-type pairs.  Factors span {raw.min():.2e} .. {raw.max():.2e}")
print(f"  (inverse to synapse count: {mean_syn.max():.1f} .. {mean_syn.min():.2f}"
      f" synapses per instance)\n")
print(f"  {'sigma':>7} {'mean':>7} {'sd':>6} {'min':>6} {'max':>6}   "
      f"{'median mean_syn of silenced':>28}")

rows = {}
for s in SIGMAS:
    fracs, med_syn = [], []
    for seed in range(N_SEEDS):
        rng = np.random.default_rng(seed)
        z = (raw + rng.normal(0, s, size=raw.shape)) <= 0.0
        fracs.append(z.mean())
        if z.any():
            med_syn.append(np.median(mean_syn[z]))
    f = np.array(fracs)
    rows[s] = f
    print(f"  {s:>7.3f} {f.mean():>6.1%} {f.std():>5.1%} "
          f"{f.min():>5.1%} {f.max():>5.1%}   {np.mean(med_syn):>26.1f}")

print(f"\n  Population median mean_syn (all 604 pairs): {np.median(mean_syn):.2f}")
print("  If the silenced median exceeds this, the deletion favours DENSE pairs.")


# ── Decile structure at each sigma ──────────────────────────────────
print("\n" + "=" * 74)
print("SILENCING PROBABILITY BY FACTOR DECILE")
print("=" * 74)
print("  Decile 1 = smallest scaling factors = DENSEST connections.\n")

dec = np.quantile(raw, np.linspace(0, 1, 11))
idx = np.clip(np.digitize(raw, dec[1:-1]), 0, 9)

hdr = "  decile   mean_syn range   " + "".join(f"{s:>9.3f}" for s in SIGMAS)
print(hdr)
for d in range(10):
    m = idx == d
    lo, hi = mean_syn[m].min(), mean_syn[m].max()
    line = f"  {d+1:>6}  {hi:>6.2f}-{lo:<6.1f}  "
    for s in SIGMAS:
        hits = tot = 0
        for seed in range(N_SEEDS):
            rng = np.random.default_rng(seed)
            z = (raw + rng.normal(0, s, size=raw.shape)) <= 0.0
            hits += z[m].sum(); tot += m.sum()
        line += f"{hits/tot:>8.1%} "
    print(line)


# ── Optional: does stability filtering bias the estimate? ────────────
import sys
if "--stability" in sys.argv:
    from flyvis.datasets.moving_bar import MovingEdge
    from flyvis.utils.activity_utils import LayerActivity

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    OVERFLOW = 1e6
    BIAS_NOISE, TC_NOISE = 0.05, 0.005
    ANGLES = [0,30,60,90,120,150,180,210,240,270,300,330]

    ds = MovingEdge(offsets=[-10,11], intensities=[0,1], speeds=[19], height=80,
                    post_pad_mode="continue", t_pre=1.0, t_post=1.0, dt=1/200,
                    angles=ANGLES)
    on_idx = [i for i,r in ds.arg_df.iterrows() if r["intensity"] == 1]
    chk = ds[on_idx[0]]
    chk = torch.as_tensor(chk, dtype=torch.float32).to(DEVICE)

    def build(seed, syn_noise):
        rng = np.random.default_rng(seed)
        net = Network()
        with torch.no_grad():
            net.nodes_bias.data += torch.tensor(
                rng.normal(0, BIAS_NOISE, size=net.nodes_bias.shape), dtype=torch.float32)
            net.nodes_time_const.data = torch.clamp(
                net.nodes_time_const.data + torch.tensor(
                    rng.normal(0, TC_NOISE, size=net.nodes_time_const.shape),
                    dtype=torch.float32), min=0.001)
            r = net.edge_params.syn_strength.raw_values
            r.data = torch.clamp(r.data + torch.tensor(
                rng.normal(0, syn_noise, size=r.shape), dtype=torch.float32), min=0.0)
            fz = float((r.data == 0).float().mean())
        return net.to(DEVICE), fz

    def stable(net):
        st = chk.unsqueeze(1) if chk.dim() == 2 else chk
        s0 = net.fade_in_state(1.0, ds.dt, st[[0]])
        out = net.simulate(st[None], ds.dt, initial_state=s0)
        o = out.cpu().numpy()
        return bool(torch.all(torch.isfinite(out)) and np.all(np.abs(o) < OVERFLOW))

    print("\n" + "=" * 74)
    print("DOES STABILITY FILTERING BIAS THE ESTIMATE?")
    print("=" * 74)
    print("  The sweep averaged frac_zeroed over ACCEPTED models. If unstable")
    print("  networks are the heavily-silenced ones, that conditions on survival")
    print("  and understates the deletion.\n")
    print(f"  {'sigma':>7} {'uncond.':>9} {'accepted':>9} {'rejected':>9} "
          f"{'n_rej':>6}  {'selection effect':>17}")

    N_TRY = 20
    for s in SIGMAS:
        acc, rej = [], []
        for seed in range(N_TRY):
            net, fz = build(seed, s)
            (acc if stable(net) else rej).append(fz)
            del net
            torch.cuda.empty_cache()
        u = rows[s][:N_TRY].mean()
        a = np.mean(acc) if acc else np.nan
        r = np.mean(rej) if rej else np.nan
        eff = (a - u) if acc else np.nan
        print(f"  {s:>7.3f} {u:>8.1%} {a:>8.1%} "
              f"{(f'{r:.1%}' if rej else '  --  '):>9} {len(rej):>6}  {eff:>+16.1%}")

    print("\n  A negative selection effect means accepted models are LESS silenced")
    print("  than a random draw: the stability filter is removing the most")
    print("  heavily-deleted networks, and the sweep's frac_zeroed understated it.")


print("\n" + "=" * 74)
print("QUOTE THESE")
print("=" * 74)
for s in SIGMAS:
    f = rows[s]
    print(f"  sigma = {s:<6} -> {f.mean():.1%} of 604 cell-type pairs silenced "
          f"(range {f.min():.1%}-{f.max():.1%}, n={N_SEEDS} seeds)")
print("\n  Unit is CELL-TYPE PAIRS (604), not synapses (1.5M).")
print("  The sweep's frac_zeroed averaged over accepted models; these are")
print("  unconditional per-seed draws.")

SILENCING FRACTION BY SIGMA — unconditional, per-seed
  604 cell-type pairs.  Factors span 6.94e-05 .. 3.73e-02
  (inverse to synapse count: 144.1 .. 0.27 synapses per instance)

    sigma    mean     sd    min    max    median mean_syn of silenced
    0.002   8.1%  1.0%  5.6% 10.3%                         11.4
    0.008  26.7%  1.7% 23.2% 31.0%                          3.2
    0.032  43.3%  2.0% 39.4% 47.5%                          2.1
    0.128  48.2%  2.2% 44.0% 53.8%                          2.0

  Population median mean_syn (all 604 pairs): 1.99
  If the silenced median exceeds this, the deletion favours DENSE pairs.

SILENCING PROBABILITY BY FACTOR DECILE
  Decile 1 = smallest scaling factors = DENSEST connections.

  decile   mean_syn range       0.002    0.008    0.032    0.128
       1  144.09-11.0       40.7%    47.1%    48.5%    48.9% 
       2   10.73-5.1        23.7%    42.9%    48.9%    50.4% 
       3    5.00-3.4        11.5%    37.2%    45.9%    48.2% 
       4    3.37-

In [20]:
# ═══════════════════════════════════════════════════════════════════
# Does the stability filter bias frac_zeroed?
#
# The sweep averaged frac_zeroed over ACCEPTED models. If unstable
# networks are the heavily-silenced ones, that conditions on survival
# and understates the deletion — and worse, the RDM spans the sweep
# reported were measured on a selected subpopulation.
#
# Test: for each sigma, build N_TRY networks from independent seeds,
# run the stability check on each, and compare frac_zeroed between
# accepted and rejected sets. Report the unconditional mean for
# reference.
#
# N_TRY is per-sigma and scaled to the observed rejection rate. At
# sigma=0.128 the sweep needed 92 seeds for 5 acceptances (~5% accept),
# so 20 seeds would yield ~1 acceptance. We use 60 there.
#
# Runtime: one forward pass per seed (~2.6 s) + network construction
# (~8 s). No stimulus loop. Roughly:
#   0.002: 25 x 11 s  =  4.5 min
#   0.008: 25 x 11 s  =  4.5 min
#   0.032: 40 x 11 s  =  7.5 min
#   0.128: 60 x 11 s  = 11.0 min
#   total ~28 minutes on a T4.
# ═══════════════════════════════════════════════════════════════════

import numpy as np
import torch
from scipy.stats import mannwhitneyu
from flyvis.network import Network
from flyvis.datasets.moving_bar import MovingEdge

SEED_TORCH = 42
np.random.seed(SEED_TORCH); torch.manual_seed(SEED_TORCH)
torch.cuda.manual_seed_all(SEED_TORCH)
torch.use_deterministic_algorithms(True)

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
OVERFLOW = 1e6
BIAS_NOISE, TC_NOISE = 0.05, 0.005
ANGLES = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]

# sigma -> number of seeds to try (scaled to observed acceptance rate)
PLAN = {0.002: 25, 0.008: 25, 0.032: 40, 0.128: 60}

ds = MovingEdge(offsets=[-10, 11], intensities=[0, 1], speeds=[19], height=80,
                post_pad_mode="continue", t_pre=1.0, t_post=1.0, dt=1/200,
                angles=ANGLES)
on_idx = [i for i, r in ds.arg_df.iterrows() if r["intensity"] == 1]
chk = torch.as_tensor(ds[on_idx[0]], dtype=torch.float32).to(DEVICE)

# Prior factors, for the density labels
_n0 = Network()
raw0 = _n0.edge_params.syn_strength.raw_values.data.cpu().numpy()
mean_syn = 0.01 / raw0          # verified: raw = 0.01 / mean_syn_count
del _n0


def build(seed, syn_noise):
    """Returns (net, frac_zeroed, median_mean_syn_of_zeroed)."""
    rng = np.random.default_rng(seed)
    net = Network()
    with torch.no_grad():
        net.nodes_bias.data += torch.tensor(
            rng.normal(0, BIAS_NOISE, size=net.nodes_bias.shape),
            dtype=torch.float32)
        net.nodes_time_const.data = torch.clamp(
            net.nodes_time_const.data + torch.tensor(
                rng.normal(0, TC_NOISE, size=net.nodes_time_const.shape),
                dtype=torch.float32), min=0.001)
        r = net.edge_params.syn_strength.raw_values
        r.data = torch.clamp(r.data + torch.tensor(
            rng.normal(0, syn_noise, size=r.shape), dtype=torch.float32), min=0.0)
        z  = (r.data == 0).cpu().numpy()
        fz = float(z.mean())
        ms = float(np.median(mean_syn[z])) if z.any() else np.nan
    return net.to(DEVICE), fz, ms


def stable(net):
    """Same check as the sweep. Does not swallow exceptions."""
    st = chk.unsqueeze(1) if chk.dim() == 2 else chk
    s0 = net.fade_in_state(1.0, ds.dt, st[[0]])
    out = net.simulate(st[None], ds.dt, initial_state=s0)
    o = out.cpu().numpy()
    return bool(torch.all(torch.isfinite(out)) and np.all(np.abs(o) < OVERFLOW))


print("=" * 78)
print("SELECTION EFFECT: is frac_zeroed different among ACCEPTED models?")
print("=" * 78)
print(f"Device: {DEVICE}\n")
print("  The sweep's frac_zeroed averaged over accepted models only. If")
print("  heavily-silenced networks are the ones that fail the stability check,")
print("  the sweep understated the deletion AND measured RDM spans on survivors.\n")

results = {}
for sig, n_try in PLAN.items():
    print(f"--- sigma = {sig}  ({n_try} seeds) " + "-" * 34)
    acc_fz, rej_fz, acc_ms, rej_ms = [], [], [], []

    for seed in range(n_try):
        net, fz, ms = build(seed, sig)
        ok = stable(net)
        (acc_fz if ok else rej_fz).append(fz)
        (acc_ms if ok else rej_ms).append(ms)
        del net
        torch.cuda.empty_cache()
        if (seed + 1) % 10 == 0:
            print(f"    {seed+1}/{n_try}  accepted {len(acc_fz)}  "
                  f"rejected {len(rej_fz)}", flush=True)

    uncond = np.array(acc_fz + rej_fz)
    a, r   = np.array(acc_fz), np.array(rej_fz)

    print(f"\n  accepted:      {len(a):3d}/{n_try}   frac_zeroed "
          f"{a.mean():.1%} ± {a.std():.1%}" if len(a) else
          f"\n  accepted:        0/{n_try}   (none)")
    print(f"  rejected:      {len(r):3d}/{n_try}   frac_zeroed "
          f"{r.mean():.1%} ± {r.std():.1%}" if len(r) else
          f"  rejected:        0/{n_try}   (none)")
    print(f"  unconditional: {len(uncond):3d}       frac_zeroed "
          f"{uncond.mean():.1%} ± {uncond.std():.1%}")

    if len(a) and len(r):
        eff = a.mean() - uncond.mean()
        u, p = mannwhitneyu(a, r, alternative='less')
        print(f"\n  selection effect (accepted - unconditional): {eff:+.2%}")
        print(f"  Mann-Whitney, accepted < rejected:  U={u:.0f}, p={p:.4f}")
        if p < 0.05:
            print("  ** Accepted models are SIGNIFICANTLY less silenced. **")
            print("     The stability filter removes heavily-deleted networks;")
            print("     the sweep's frac_zeroed and RDM spans are conditioned on")
            print("     survival.")
        else:
            print("  -> No detectable selection on frac_zeroed.")
        if not (np.isnan(np.nanmean(acc_ms)) or np.isnan(np.nanmean(rej_ms))):
            print(f"  median mean_syn of silenced pairs: "
                  f"accepted {np.nanmean(acc_ms):.1f}, rejected {np.nanmean(rej_ms):.1f}")
    elif len(r) == 0:
        print("\n  All seeds accepted. No selection possible at this sigma.")
    else:
        print("\n  All seeds rejected. Cannot compare.")

    results[sig] = dict(acc=a, rej=r, uncond=uncond)
    print()


# ── SUMMARY ─────────────────────────────────────────────────────────
print("=" * 78)
print("SUMMARY")
print("=" * 78)
print(f"  {'sigma':>7} {'accept rate':>12} {'uncond.':>9} {'accepted':>9} "
      f"{'rejected':>9} {'effect':>9}")
for sig in PLAN:
    d = results[sig]
    a, r, u = d['acc'], d['rej'], d['uncond']
    rate = len(a) / len(u)
    am = f"{a.mean():.1%}" if len(a) else "  --  "
    rm = f"{r.mean():.1%}" if len(r) else "  --  "
    ef = f"{a.mean()-u.mean():+.2%}" if len(a) else "  --  "
    print(f"  {sig:>7.3f} {rate:>11.0%} {u.mean():>8.1%} {am:>9} {rm:>9} {ef:>9}")

print("""
  INTERPRETATION

  Effect ~ 0        The stability filter is orthogonal to how much was
                    silenced. The sweep's frac_zeroed is an unbiased
                    estimate, and its RDM spans were not measured on a
                    selected subpopulation.

  Effect negative   Accepted models are less silenced than random draws.
                    The sweep understated the deletion, and the RDM spans
                    it reported belong to the networks that happened to
                    survive — a biased sample. Report the unconditional
                    fractions and say the spans are conditional.

  Effect positive   Surprising. Would mean deletion stabilises the network,
                    which is possible if the silenced pairs are excitatory
                    and their removal reduces runaway gain. Worth checking
                    the E/I composition of the zeroed pairs before writing
                    anything.
""")

SELECTION EFFECT: is frac_zeroed different among ACCEPTED models?
Device: cuda

  The sweep's frac_zeroed averaged over accepted models only. If
  heavily-silenced networks are the ones that fail the stability check,
  the sweep understated the deletion AND measured RDM spans on survivors.

--- sigma = 0.002  (25 seeds) ----------------------------------
    10/25  accepted 10  rejected 0
    20/25  accepted 20  rejected 0

  accepted:       25/25   frac_zeroed 8.5% ± 0.9%
  rejected:        0/25   (none)
  unconditional:  25       frac_zeroed 8.5% ± 0.9%

  All seeds accepted. No selection possible at this sigma.

--- sigma = 0.008  (25 seeds) ----------------------------------
    10/25  accepted 10  rejected 0
    20/25  accepted 20  rejected 0

  accepted:       25/25   frac_zeroed 26.8% ± 1.7%
  rejected:        0/25   (none)
  unconditional:  25       frac_zeroed 26.8% ± 1.7%

  All seeds accepted. No selection possible at this sigma.

--- sigma = 0.032  (40 seeds) --------------

## Download Files

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Download results
# ═══════════════════════════════════════════════════════════════════
from google.colab import files
files.download("/content/results/exp4_sweep.npz")
files.download("/content/figures/exp4_sweep.png")

## Results

*All results in this section are from `exp4_sweep.npz` (bias axis) and
`exp4_sweep_syn.npz` (synapse axis), N = 5 models per condition.*

**Note on ensemble size.** This sweep uses N = 5 per condition; Experiment 4 used
N = 50. At the shared configuration (BIAS\_NOISE = 0.05, SYN\_STRENGTH\_NOISE = 0.002)
the sweep reports span/floor = 0.05× against Experiment 4's 0.09×. Averaging fewer RDMs
suppresses less noise, and the guard's floor — which uses max\|response\| across all
models — does not credit ensemble size. This is one of several reasons the 10× margin is
a heuristic rather than a derived threshold. The per-model cross-check is unaffected:
τ = 1.0000 at every noise level in every condition, in both sweeps.

### Bias noise destroys the geometry it was meant to reveal

| σ (BIAS\_NOISE) | CC span | CC floor | ratio | CC \|resp\| |
|---|---|---|---|---|
| 0.05 | 1.03×10⁻⁸ | 1.90×10⁻⁷ | 0.05× | 1.59 |
| 0.2 | 8.71×10⁻⁹ | 2.24×10⁻⁷ | 0.04× | 1.88 |
| 0.8 | 3.23×10⁻⁹ | 3.73×10⁻⁷ | 0.01× | 3.13 |
| 3.2 | 2.53×10⁻¹⁰ | 1.32×10⁻⁶ | 0.00× | 11.08 |

**CC's span falls monotonically by a factor of forty while response magnitude rises
sevenfold.** `nodes_bias` is a per-node additive constant: a large common offset drives
all twelve population vectors toward the same bias-dominated direction. Cosine distance
is scale-invariant — it does not see that the vectors grew, only that the angle between
them shrank. Bias noise does not fail to reveal directional geometry. It destroys it.
The axis is foreclosed, with a mechanism.

### Synapse-strength noise moves in the right direction and still fails

`edges_syn_strength` is the parameter the connectome constrains. Sweeping it, with
BIAS\_NOISE fixed at 0.05:

| σ (SYN\_STRENGTH\_NOISE) | CC span | ratio | CC \|resp\| | rejections | pairs silenced |
|---|---|---|---|---|---|
| 0.002 | 1.03×10⁻⁸ | 0.05× | 1.59 | 0 | **8.1%** |
| 0.008 | 4.07×10⁻⁷ | **1.20×** | 2.85 | 0 | 26.7% |
| 0.032 | 3.23×10⁻⁵ | 1.10× | 245.55 | 2 | 43.3% |
| 0.128 | 1.92×10⁻⁷ | 0.00× | 498.03 | 87 | 48.2% |

*Silencing fractions are unconditional per-seed means (n = 50), not the accepted-model
averages the sweep's `frac_zeroed` originally reported.*

CC's span *rises* — 1.03×10⁻⁸ → 4.07×10⁻⁷, a factor of forty — and the ratio reaches
1.20×, twenty-four times better than bias noise ever achieved. Then it stalls. It never
approaches the threshold. Meanwhile response magnitude reaches 245 and then 498, and
rejections climb to 87 per 5 acceptances.

**Two diagnostics locate where the regime ends.**

*The deletion loses its density selectivity.* At σ = 0.002 the lowest-factor decile is
silenced 40.7% of the time and the top four deciles never. At σ = 0.128 every decile is
silenced at ~48%, because the noise exceeds even the largest scaling factor and the clamp
becomes a coin flip.

*The acceptance rate collapses.* Under the stability check: **100%, 100%, 68%, 5%** across
the four levels (n = 25, 25, 40, 60 seeds). At σ = 0.128, **57 of 60 seeds are dynamically
unstable** — the models the sweep evaluated there are a 5% tail, not a typical draw.

**By the time the perturbation is large enough to be a candidate for clearing the guard,
the network is not a sparsified connectome. It is half a connectome, deleted at random, and
almost never stable.**

*(The σ = 0.128 level did not complete for the Rand-sign condition; the run was interrupted.
CC and Rand-syn values are reported above. The level is already outside any defensible
regime — 87 and 96 rejections, responses at 498 and 3508, 48% of pairs silenced — so the
missing row does not affect the conclusion.)*

### The stability filter does not bias the silencing estimate

The sweep averaged `frac_zeroed` over *accepted* models. If heavily-silenced networks were
the ones failing the stability check, both the fractions and the RDM spans would be
conditioned on survival. They are not.

| σ | accept rate | unconditional | accepted | rejected | Mann–Whitney p |
|---|---|---|---|---|---|
| 0.002 | 100% | 8.5% | 8.5% | — | — |
| 0.008 | 100% | 26.8% | 26.8% | — | — |
| 0.032 | 68% | 43.5% | 43.3% | 43.7% | 0.38 |
| 0.128 | 5% | 48.1% | 48.5% | 48.1% | 0.57 |

At the two lowest levels no seed was rejected. At σ = 0.032 accepted and rejected models are
silenced at the same rate, with identical median synapse counts among silenced pairs (2.1
vs 2.1). The +0.34% at σ = 0.128 rests on 3 accepted models and is not interpretable.

**The reported silencing fractions and RDM spans are therefore not conditioned on survival.**

### The control: the only resolvable point is a saturation artifact

The sole condition that ever cleared the guard, in either sweep, is Rand-sign — with E/I
identity scrambled:

| sweep | σ | ratio | r(circ) | \|resp\| | CC \|resp\| at same σ |
|---|---|---|---|---|---|
| bias | 0.8 | 15.28× | **+0.495** | 15.37 | 3.13 |
| synapse | 0.032 | 27.80× | **+0.499** | 193.40 | 245.55 |

Two unrelated perturbation axes produce the same circularity to three decimal places, in
the condition whose excitatory/inhibitory balance has been destroyed, at response
magnitudes an order of magnitude above CC's in the bias sweep. This is not a wiring prior.
It is what a network without E/I balance does when driven hard enough to saturate: large
direction-dependent responses whose geometry is dominated by which cells blow up. Its RDM
clears the floor because its responses are enormous, not because its wiring is informative.

Without this control the sweep would have been a mechanism for manufacturing whichever
answer was sought: raise the noise until something becomes resolvable, then report it.

### The baseline condition was never what it was described as

A diagnostic added for this sweep reports the fraction of cell-type pairs whose
`edges_syn_strength` is clamped to exactly zero. The parameter is one scaling factor per
(source, target) cell-type pair — 604 in total — initialized as **`0.01 / mean_syn_count`**,
*inverse* to synapse count. Gaussian noise at σ = 0.002 exceeds the smallest factors, which
belong to the *densest* connections.

**At the Flyvis prior — the configuration every "untrained CC" model in this work has used,
including all of Experiment 4 — 8.1% of cell-type pairs are silenced (range 5.6–10.3%,
n = 50 seeds), and 40.7% of the densest decile.** An entire cell-type-to-cell-type
projection is deleted, not a few synapses attenuated. This does not alter any conclusion
(the guard was already refusing to report), but the baseline condition was sparsified along
its densest projections, not merely perturbed, and this belongs in the Methods of both
experiments.

## Discussion

- **Untrained connectome-constrained networks have no measurable representational
  geometry, along either perturbation axis, across four orders of magnitude, within the
  regime where they remain connectome-constrained networks.** Bias noise makes the
  population vectors parallel. Synapse-strength noise separates them but inflates
  responses, silences up to half the cell-type connections, and destabilizes the network
  (acceptance rate 100% → 5%) before the RDM clears its own floor.

- **The original claim is not merely unsupported; it is unrescuable by its own proposed
  remedy.** The limitations section of the original Experiment 4